In [4]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ==========================================
# 0. INSTÄLLNINGAR & ENHETSHANTERING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Tränar på: {device}")

# ==========================================
# 1. MODELL-ARKITEKTURER & FUNKTIONER
# ==========================================

# Modell 1: Enkel LSTM Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=64):
        super(LSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim

        self.encoder = nn.LSTM(input_size=n_features, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.decoder = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        encoded = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(encoded)
        reconstructed = self.output_layer(decoded)
        return reconstructed

# Modell 2: Djup LSTM Autoencoder (Uppfyller kravet på en andra arkitektur)
class DeepLSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, hidden_dim=64, num_layers=2):
        super(DeepLSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.encoder = nn.LSTM(input_size=n_features, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.decoder = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.output_layer = nn.Linear(hidden_dim, n_features)

    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        encoded = hidden[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _ = self.decoder(encoded)
        reconstructed = self.output_layer(decoded)
        return reconstructed

# Funktion för att skapa sekvenser (Sliding Windows)
def create_sequences(data, seq_length):
    xs = []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
    return np.array(xs)

# Funktion för att beräkna rekonstruktionsfel (MAE) för utvärdering
def get_reconstruction_errors(model, dataloader):
    model.eval()
    errors = []
    with torch.no_grad():
        for (batch_x,) in dataloader:
            batch_x = batch_x.to(device)
            reconstructed = model(batch_x)
            # Beräkna MAE per sekvens
            loss_per_sequence = torch.mean(torch.abs(reconstructed - batch_x), dim=(1, 2))
            errors.extend(loss_per_sequence.cpu().numpy())
    return np.array(errors)


# ==========================================
# 2. FÖRBEREDA TRÄNINGSDATA
# ==========================================
print("\n--- Laddar och förbereder träningsdata ---")
# VIKTIGT: Fyll i rätt sökväg till din träningsdata
train_data_path = "/mnt/c/Users/linne/Skola/Deep-Learning-DV2646/Uppgift 2/Train_data.csv"
df_train = pd.read_csv(train_data_path)

# VIKTIGT: Filtrera fram BARA normal data för träningen!
# Exempel: df_train = df_train[df_train['label_column'] == 'normal']

df_train_numeric = df_train.select_dtypes(include=[np.number]).dropna()

# Skala datan
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df_train_numeric)

# Skapa sekvenser
seq_len = 10
X_seq = create_sequences(data_scaled, seq_len)
n_features = X_seq.shape[2] 
print(f"Träningsdata formaterad. Input shape är: (Batch, {seq_len}, {n_features})")

# Dataloader
train_tensor = torch.tensor(X_seq, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=64, shuffle=True)


# ==========================================
# 3. INITIERA MODELL OCH TRÄNING
# ==========================================
print("\n--- Startar träning ---")
# Här väljer du om du vill träna Modell 1 (LSTMAutoencoder) eller Modell 2 (DeepLSTMAutoencoder)
model = LSTMAutoencoder(seq_len, n_features).to(device)
criterion = nn.L1Loss() # MAE
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 50

for epoch in range(epochs):
    model.train() 
    train_loss = 0
    for (batch_x,) in train_loader:
        batch_x = batch_x.to(device)
        
        optimizer.zero_grad()
        reconstructed = model(batch_x)
        loss = criterion(reconstructed, batch_x)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_loss = train_loss / len(train_loader)
    # Skriver ut var 10:e epoch för att inte spamma konsolen
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}')


# ==========================================
# 4. BERÄKNA TRÖSKELVÄRDE (THRESHOLD)
# ==========================================
print("\n--- Beräknar tröskelvärde för anomalier ---")
train_errors = get_reconstruction_errors(model, train_loader)
# Sätter tröskeln till 95:e percentilen av felen på normal data
threshold = np.percentile(train_errors, 95)
print(f"Tröskelvärde (Threshold) satt till: {threshold:.4f}")


# ==========================================
# 5. FÖRBEREDA TESTDATA & UTVÄRDERING
# ==========================================
print("\n--- Laddar och utvärderar testdata ---")
# VIKTIGT: Fyll i rätt sökväg till din testdata
test_data_path = "/mnt/c/Users/linne/Skola/Deep-Learning-DV2646/Uppgift 2/Test_data.csv" 
df_test = pd.read_csv(test_data_path)

# VIKTIGT: Ändra 'label_column_name' till vad kolumnen för anomali/normal heter i din fil!
# Vi antar att 0 är Normal och 1 är Anomali/Attack
true_labels = df_test['label_column_name'].apply(lambda x: 0 if x == 'normal' else 1).values

# Filtrera ut numeriska kolumner
df_test_numeric = df_test.select_dtypes(include=[np.number]).dropna()

# VIKTIGT: Använd transform() (inte fit_transform) så att testdatan skalas enligt träningsdatan
test_scaled = scaler.transform(df_test_numeric)

# Skapa sekvenser
X_test_seq = create_sequences(test_scaled, seq_len)

# Skapa test dataloader (VIKTIGT: shuffle=False så vi kan matcha mot etiketterna)
test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
test_loader = DataLoader(TensorDataset(test_tensor), batch_size=64, shuffle=False)

# Trimma labels så de matchar sekvensernas längd (första 'seq_len' raderna "försvinner" när fönstret skapas)
true_labels_seq = true_labels[seq_len:]

# ==========================================
# 6. ANOMALIDETEKTION & METRIKER
# ==========================================
test_errors = get_reconstruction_errors(model, test_loader)

# Förutspå anomalier: Om felet är större än tröskeln -> 1 (Anomali), annars 0 (Normal)
predictions = (test_errors > threshold).astype(int)

# Beräkna mätvärden
precision = precision_score(true_labels_seq, predictions)
recall = recall_score(true_labels_seq, predictions)
f1 = f1_score(true_labels_seq, predictions)
roc_auc = roc_auc_score(true_labels_seq, test_errors) # Använder det faktiska felet som "sannolikhet"

print("\n==================================")
print("       SLUTGILTIGA RESULTAT       ")
print("==================================")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Tränar på: cpu

--- Laddar och förbereder träningsdata ---
Träningsdata formaterad. Input shape är: (Batch, 10, 38)

--- Startar träning ---
Epoch 1/50, Loss: 0.112949
Epoch 10/50, Loss: 0.103307
Epoch 20/50, Loss: 0.063912
Epoch 30/50, Loss: 0.047912
Epoch 40/50, Loss: 0.043609
Epoch 50/50, Loss: 0.032124

--- Beräknar tröskelvärde för anomalier ---
Tröskelvärde (Threshold) satt till: 0.0513

--- Laddar och utvärderar testdata ---


KeyError: 'label_column_name'